In [1]:
from pathlib import Path
import sys
import json
from datetime import datetime
from json import JSONDecodeError

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from Llm.llm_loader import LLM
import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv(project_root / '.env')

True

In [ ]:

model_name = 'gpt-3.5-turbo'
Answer_llm = LLM(model_name = model_name, provider= "openai")
dataset = 'MMQA'
extract_table_prompt_path = project_root / 'Templates' / 'zero_shot' / 'extract_relevant_tables.txt'
extract_column_prompt_path = project_root / 'Templates' / 'zero_shot' / 'extract_relevant_columns.txt'
logs_dir = project_root / 'Logs' / model_name
logs_dir.mkdir(exist_ok=True)
run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
log_path = logs_dir / f'zero_shot_table_column_{dataset}_{run_id}.json'

In [3]:
if dataset == 'MMQA':
    datasets = [
        (
            'two_table',
            project_root / 'Data' / dataset / 'Synthesized_two_table_with_spider_db_id.json',
            pd.read_json(project_root / 'Data' / dataset / 'Synthesized_two_table_with_spider_db_id.json')
        ),
        (
            'three_table',
            project_root / 'Data' / dataset / 'Synthesized_three_table_with_spider_db_id.json',
            pd.read_json(project_root / 'Data' / dataset / 'Synthesized_three_table_with_spider_db_id.json')
        ),
    ]
else:
    raise ValueError(f'Unsupported dataset: {dataset}')

In [4]:
def append_log_entry(log_records, data_split, row, relevant_table_list,relevant_columns):
    log_records.append(
        {
            'model': Answer_llm.model_name,
            'provider': Answer_llm.provider,
            'template_name': 'table2column',
            'id': f"{dataset}-{data_split}-{row['id_']}",
            'spider_db_id': row['Spider_db_id'],
            'question': row['Question'],
            'relevant_tables': relevant_table_list,
            'relevant_columns' : relevant_columns
        }
    )
    log_path.write_text(json.dumps(log_records, ensure_ascii=False, indent=2), encoding='utf-8')

In [5]:
extract_table_prompt_template = extract_table_prompt_path.read_text(encoding='utf-8').strip()
extract_column_prompt_template = extract_column_prompt_path.read_text(encoding='utf-8').strip()
log_records = []
log_path.write_text(json.dumps(log_records, ensure_ascii=False, indent=2), encoding='utf-8')

for data_split, data_source, df in datasets:
    for _, row in tqdm(df.iterrows(), total=len(df), desc=data_split):
        schema_path = project_root / 'Data' / 'spider' / 'database' / row['Spider_db_id'] / 'schema description.csv'
        total_schema_df = pd.read_csv(schema_path)
        extract_table_prompt = (
            extract_table_prompt_template
            .replace('{DATABASE_SCHEMA}', total_schema_df.to_markdown())
            .replace('{QUESTION}', row['Question'])
            .replace('{HINT}', 'No hint')
        )
        relevant_tables = Answer_llm.query(extract_table_prompt)
        try:
            relevant_table_list = json.loads(relevant_tables.strip())
        except JSONDecodeError:
            relevant_table_list = []

        if relevant_table_list:
            relevant_schema_df = total_schema_df[total_schema_df['table_name'].isin(relevant_table_list)]
        else:
            relevant_schema_df = total_schema_df

        extract_column_prompt = (
            extract_column_prompt_template
            .replace('{DATABASE_SCHEMA}', relevant_schema_df.to_markdown())
            .replace('{QUESTION}', row['Question'])
            .replace('{HINT}', 'No hint')
        )
        relevant_columns = Answer_llm.query(extract_column_prompt)

        append_log_entry(log_records, data_split, row, relevant_table_list,relevant_columns)

print(f'All responses have been saved to {log_path}')






three_table: 100%|██████████| 721/721 [20:17<00:00,  1.69s/it]

All responses have been saved to /home/xubeiyu/projects/Schemalinking/Logs/gpt-3.5-turbo/zero_shot_table_column_MMQA_20260304_125800.json
